# Module 0: Welcome & Setup

**In the workshop (20 minutes):** welcome and expectations, then we start the installation
together. Follow the steps in the README (or run `bash bin/install.sh`), then run the two check
cells below. The downloads take a while; let them run in the background while Module 1 starts
with the concepts.

If you have already installed everything before the workshop, just run the two cells to check.

---

In [ ]:
# Environment check: run this cell and fix anything marked ✗ before continuing
import importlib
import sys
import warnings
from os import getenv

from tqdm import TqdmExperimentalWarning

warnings.filterwarnings('ignore', category=TqdmExperimentalWarning)

REQUIRED_PYTHON = (3, 11)
REQUIRED_PACKAGES = [
    'chromadb',
    'sentence_transformers',
    'gradio',
    'requests',
    'numpy',
    'pypdf',
    'tqdm',
    'yake',
    'docx',
    'matplotlib',
    'umap',
]

print('Python version:', sys.version)
if sys.version_info >= REQUIRED_PYTHON:
    print(f'  ✓ Python {REQUIRED_PYTHON[0]}.{REQUIRED_PYTHON[1]}+ detected')
else:
    print(f'  ✗ Python {REQUIRED_PYTHON[0]}.{REQUIRED_PYTHON[1]}+ required, please update')

print('\nPackages:')
for pkg in REQUIRED_PACKAGES:
    try:
        importlib.import_module(pkg)
        print(f'  ✓ {pkg}')
    except ImportError:
        print(f'  ✗ {pkg} missing, run: uv sync')

print('\nragsst package:')
try:
    importlib.import_module('ragsst')
    print('  ✓ ragsst installed')
except ImportError:
    print('  ✗ ragsst not found, run: uv sync')

print('\nOllama:')
REQUIRED_LLM = 'llama3.2:3b'
try:
    import requests
    ollama_host = getenv('OLLAMA_HOST', 'http://localhost:11434')
    r = requests.get(f'{ollama_host}/api/tags', timeout=3)
    models = [m['name'] for m in r.json().get('models', [])]
    print('  ✓ Ollama running')
    if REQUIRED_LLM in models:
        print(f'  ✓ {REQUIRED_LLM} available')
    else:
        print(f'  ✗ {REQUIRED_LLM} missing, run: ollama pull {REQUIRED_LLM}')
except Exception:
    print('  ✗ Ollama not reachable, run: ollama serve')

print('\nAll done! Fix any ✗ items above before proceeding.')

### Download the embedding and re-ranking models

Every module loads one of these models from Hugging Face. Downloading them now (about 700 MB in
total) means nobody waits for a download in the middle of an exercise.

In [ ]:
import warnings

warnings.filterwarnings('ignore')

from sentence_transformers import CrossEncoder, SentenceTransformer

EMBEDDING_MODELS = [
    'all-MiniLM-L6-v2',            # Modules 1, 2
    'multi-qa-MiniLM-L6-cos-v1',   # Module 2
    'multi-qa-mpnet-base-cos-v1',  # Modules 3 to 7 (RAGTool default)
]
RERANKER = 'cross-encoder/ms-marco-MiniLM-L-6-v2'  # Module 6

for name in EMBEDDING_MODELS:
    SentenceTransformer(name)
    print(f'  ✓ {name}')
CrossEncoder(RERANKER)
print(f'  ✓ {RERANKER}')
print('\nAll models cached.')

## What you will build today

By the end of this workshop you will have a working local RAG system:
load your own documents, embed them into a vector database, and query them
through a chat interface powered by a local LLM.

<img src="../images/rag_architecture.svg" width="600" />

More importantly, you will understand every step of the pipeline well enough
to tune and debug it yourself.

### The technology stack

| Layer | Tool | Why |
|-------|------|-----|
| LLM | Ollama (llama3.2:3b) | Runs locally, no API key, fast enough for workshops |
| Embeddings | Sentence Transformers | Open weights, many models, easy to use |
| Vector DB | ChromaDB | Embeds well in notebooks, persistent, no server needed |
| Interface | Gradio | One-liner to launch a web UI from Python |

### How this workshop is structured

The first half (Modules 1 to 4) is about *understanding*:
what RAG is, how embeddings work, how chunking affects quality, and how to wire it all together.

The second half (Modules 5 to 7) is about *measuring and improving*:
first we measure how good the basic pipeline is, then we try techniques to improve it
and check with numbers whether they actually helped, and finally you apply this to
documents of your choice.

### What RAG is NOT

Before diving in, a few things worth clearing up:

- RAG does not make the model smarter. It gives the model *information to read*
- RAG does not guarantee correct answers. A poorly chunked corpus will still produce bad answers
- RAG is not always the right tool. For very short documents, just put them in the prompt
- RAG is not magic. It is a well-understood pipeline with well-understood failure modes

---

## Your expectations (5 minutes)

Take a sheet of paper (or a text file) and draw three columns:

| I already know... | I want to find out... | I learned... |
|---|---|---|
| | | |

Fill in the first two columns now, about RAG, LLMs, or anything related. Leave the third one
empty. We come back to it at the end of the day in Module 7.

---

## Quick demo

The trainer shows the finished app on the projector. This is where you will be by the end of
Module 4. Once your installation has finished, you can run the cell below yourself; close the
app when you are done, we rebuild it step by step during the workshop.

In [ ]:
# Needs Ollama running and llama3.2:3b pulled (see the check cell above)

from ragsst.interface import make_interface
from ragsst.ragtool import RAGTool

tool = RAGTool(data_path='../data/sample_docs', collection_name='workshop_docs')
tool.setup_vec_store()
gui = make_interface(tool)
gui.launch()